In [1]:
from typing import Tuple
from pathlib import Path

import torch
from torch import nn
from torch import functional as F

device = ""
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

def load_data(p: Path | str, train_data_len, val_data_len) -> Tuple[bytes, bytes]:
    with open(p, "r") as fp:
        text = fp.read()
    train_data = text[:train_data_len]
    val_data = text[train_data_len:train_data_len+val_data_len]
    return train_data.encode("utf-8"), val_data.encode("utf-8")

train_data, val_data = load_data("tiny-shakespeare.txt", 1000000, 100000)


In [2]:
from collections import Counter

class BPETokenizer:
    def __init__(self):
        self.vocab = [bytes([i]) for i in range(256)]
        self.merges = {}

    def merge_all(self, ids: list[int], pair, new_id):
        i = 0
        res = []
        while i < len(ids):
            if i+1 < len(ids) and (ids[i], ids[i+1]) == pair:
                res.append(new_id)
                i += 2
            else:
                res.append(ids[i])
                i += 1
        return res

    def train(self, data: bytes, vocab_size: int):
        ids = list(data)
        print(len(ids))
        while len(self.vocab) < vocab_size and len(ids) > 1:
            stats = Counter(zip(ids, ids[1:]))
            pair = max(stats, key=lambda p: stats[p])
            new_id = len(self.vocab)
            self.merges[pair] = new_id
            self.vocab.append(self.vocab[pair[0]]+self.vocab[pair[1]])
            ids = self.merge_all(ids, pair, new_id)

    def encode(self, data: bytes):
        ids = list(data)
        while len(ids) > 1:
            pairs = set(zip(ids, ids[1:]))
            p = min(pairs, key=lambda p: self.merges.get(p, 10**9))
            if p not in self.merges:
                break
            ids = self.merge_all(ids, p, self.merges[p])
        return ids

    def decode(self, ids: list[int]):
        return b"".join(map(lambda id: self.vocab[id], ids))

tokenizer = BPETokenizer()
tokenizer.train(train_data, 500)
print(tokenizer.vocab)
print(tokenizer.encode(b"for"))
print(tokenizer.decode([299, 299]))

1000000
[b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f', b'\x10', b'\x11', b'\x12', b'\x13', b'\x14', b'\x15', b'\x16', b'\x17', b'\x18', b'\x19', b'\x1a', b'\x1b', b'\x1c', b'\x1d', b'\x1e', b'\x1f', b' ', b'!', b'"', b'#', b'$', b'%', b'&', b"'", b'(', b')', b'*', b'+', b',', b'-', b'.', b'/', b'0', b'1', b'2', b'3', b'4', b'5', b'6', b'7', b'8', b'9', b':', b';', b'<', b'=', b'>', b'?', b'@', b'A', b'B', b'C', b'D', b'E', b'F', b'G', b'H', b'I', b'J', b'K', b'L', b'M', b'N', b'O', b'P', b'Q', b'R', b'S', b'T', b'U', b'V', b'W', b'X', b'Y', b'Z', b'[', b'\\', b']', b'^', b'_', b'`', b'a', b'b', b'c', b'd', b'e', b'f', b'g', b'h', b'i', b'j', b'k', b'l', b'm', b'n', b'o', b'p', b'q', b'r', b's', b't', b'u', b'v', b'w', b'x', b'y', b'z', b'{', b'|', b'}', b'~', b'\x7f', b'\x80', b'\x81', b'\x82', b'\x83', b'\x84', b'\x85', b'\x86', b'\x87', b'\x88', b'\x89', b'\x8a', b'\x8b', b'\x8c', b'\x8d', b'\